## 1. Business Question

**Goal:** Predict whether a bank customer will churn (leave the financial institution), enabling proactive retention efforts for at-risk customers.

**Problem type:** Binary classification

**Why this matters:** Acquiring a new customer typically costs more than retaining an existing one. Identifying customers likely to churn allows the business to intervene early — through targeted outreach, incentives, or service improvements — before the customer leaves.

**Evaluation metrics:**
Churned customers are expected to represent the minority class, making accuracy an unreliable metric on its own. This project prioritizes:
- **F1 Score** (primary) — balances precision and recall, which matters here since both false positives (wasted retention spend) and false negatives (missed at-risk customers) carry real costs. The required test F1 score for this project is at least 0.59.
- **AUC-ROC** (secondary) — evaluates the model's ability to distinguish churners from non-churners across all classification thresholds, independent of any single cutoff.

## 2. Data Description

The dataset contains 10,000 customer records with 14 columns, provided by TripleTen for this project.

| Column | Description | Type |
|---|---|---|
| RowNumber | Row index — not a customer attribute | Unusable (identifier) |
| CustomerId | Unique customer identifier | Unusable (identifier) |
| Surname | Customer's last name | Unusable (high-cardinality metadata) |
| CreditScore | Customer's credit score | Feature |
| Geography | Customer's country/region | Feature |
| Gender | Customer's gender | Feature |
| Age | Customer's age | Feature |
| Tenure | Number of years as a bank customer | Feature |
| Balance | Account balance | Feature |
| NumOfProducts | Number of bank products the customer uses | Feature |
| HasCrCard | Whether the customer has a credit card (1 = yes, 0 = no) | Feature |
| IsActiveMember | Whether the customer is an active member (1 = yes, 0 = no) | Feature |
| EstimatedSalary | Customer's estimated salary | Feature |
| Exited | Whether the customer churned (1 = yes, 0 = no) | **Target** |

**Feature types:**

*Categorical:*
- `Geography` — nominal (needs one-hot encoding before training)
- `Gender` — nominal (needs one-hot encoding before training)
- `HasCrCard` — binary (1 = yes, 0 = no)
- `IsActiveMember` — binary (1 = yes, 0 = no)
- `Exited` (target) — binary (1 = churned, 0 = retained)
- `Surname` — high-cardinality, unusable (see Section 5)

*Numerical:*
- `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `EstimatedSalary`

*Identifiers (not features):*
- `RowNumber`, `CustomerId` — unusable (see Section 5)

## 3. Load and Inspect the Data

**Data source:** Provided by TripleTen as part of the Data Science bootcamp program coursework. This dataset is used for educational purposes and is not publicly redistributed.

**Unit of observation:** Each row represents one bank customer.

Before any cleaning or analysis, this section loads the raw dataset and performs an initial inspection to understand its structure, size, and data types — setting up the deeper data quality checks in Section 4.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'Churn.csv'

df = pd.read_csv(data_path)

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.sample(5, random_state=12345)

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

**Initial observations:**
- The dataset contains **10,000 rows** and **14 columns**, matching the structure described in Section 2.
- Data types are largely consistent with expectations:
  - `CreditScore`, `Age`, `NumOfProducts`, `HasCrCard`, `IsActiveMember`, and `Exited` are `int64`.
  - `Tenure`, `Balance`, and `EstimatedSalary` are `float64`.
  - `Surname`, `Geography`, and `Gender` are stored as strings (`object`); `Geography` and `Gender` will be one-hot encoded ahead of modeling.
  - `RowNumber` and `CustomerId` are also `int64`, but as noted in Section 2 these are identifiers with no predictive value and will be dropped rather than treated as numeric features.

## 4. Data Quality Audit

Before analyzing patterns in the data, this section verifies its integrity across four checks: missing values, duplicate records, sensible feature value ranges, and consistency of categorical values. Issues found here will determine the cleaning steps applied in later sections.

**Missing values**

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
})

missing[missing['missing_count'] > 0]

**Duplicate records**

Because identifiers like `CustomerId` can hide otherwise duplicated records, duplicates are checked both including and excluding identifier columns.

In [ ]:
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate customer IDs:', df['CustomerId'].duplicated().sum())

In [ ]:
comparison_columns = df.columns.drop(['RowNumber', 'CustomerId'])

df.duplicated(subset=comparison_columns).sum()

**Sensible feature value ranges**

Checking numerical columns for values outside a plausible range (e.g. negative age, negative balance) that would indicate data entry errors.

In [ ]:
df[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']].agg(['min', 'max'])

**Categorical value consistency**

Inspecting the unique values in categorical columns to catch inconsistent labeling (e.g. typos, mismatched casing, unexpected categories).

In [ ]:
for column in ['Geography', 'Gender', 'NumOfProducts',
               'HasCrCard', 'IsActiveMember', 'Exited']:
    print(f'\n{column}')
    print(df[column].value_counts(dropna=False))

## 5. Document Usable Features

As flagged in Section 2, `RowNumber`, `CustomerId`, and `Surname` carry no predictive value and will be excluded from modeling. They are retained in the dataset through the remainder of this EDA but will be dropped during preprocessing.

**Final feature set for modeling:**
- Numerical: `CreditScore`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `EstimatedSalary`
- Categorical (to be encoded): `Geography`, `Gender`
- Binary: `HasCrCard`, `IsActiveMember`
- Target: `Exited`

## 6. Investigate Missing Tenure

Section 4 identified 909 missing values in `Tenure` (9.09% of records). Before deciding how to handle them — dropping, imputing, or treating missingness itself as informative — this section investigates whether the missing values occur randomly or are related to other variables in the data.

**Missingness rate**

In [ ]:
df['Tenure'].isna().value_counts()


In [ ]:
df['Tenure'].isna().mean() 

**Missingness vs. churn**

In [ ]:
df.assign(TenureMissing=df['Tenure'].isna()).groupby(
    'TenureMissing'
)['Exited'].agg(['count', 'mean'])

**Missingness vs. Geography**

In [ ]:
pd.crosstab(df['Geography'],
            df['Tenure'].isna(),
            normalize='index'
).mul(100).round(2)

**Findings:**

No obvious association was found between missing `Tenure` values and either churn or geography. The source of the missingness remains unknown, but these checks do not indicate that missingness itself contains useful predictive information.

**Implications for preprocessing:**
- These checks do not justify adding a `Tenure`-missing indicator at this stage.
- Dropping these rows would discard ~9% of the data for no clear benefit.
- **Median imputation** is a reasonable choice, since `Tenure` is roughly uniform between 0 and 10 with no extreme skew or outliers.

## 7. Examine Target Imbalance

Section 1 anticipated that churned customers would represent the minority class. This section quantifies that imbalance by examining the distribution of the target variable (`Exited`), confirming the choice of F1 and AUC-ROC over accuracy as evaluation metrics.

In [ ]:
target_distribution = df['Exited'].value_counts()
target_percentage = df['Exited'].value_counts(normalize=True).mul(100)

print(target_distribution)
print(target_percentage)

In [ ]:
sns.countplot(data=df, x='Exited')
plt.title('Customer Churn Distribution')
plt.show()

**Findings:**

Approximately 79.63% of customers stayed, while the remainder churned — confirming the class imbalance anticipated in Section 1.

This imbalance makes accuracy a misleading metric: a model that naively predicted "stayed" for every customer would achieve ~79.63% accuracy while providing zero value in identifying at-risk customers. This reinforces the earlier decision to prioritize F1 (which accounts for both false positives and false negatives) and AUC-ROC (which evaluates performance across classification thresholds) instead.

## 8. Explore Numerical Features

This section examines the distribution of each numerical feature individually to understand their shape, spread, and any irregularities that may inform preprocessing or feature engineering decisions ahead of modeling.

In [ ]:
numeric_columns = [
    'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary'
]

In [ ]:
df[numeric_columns].hist(bins=20, figsize=(12, 8))
plt.tight_layout()
plt.show()

**Findings:**

- **CreditScore** is approximately bell-shaped and centered around 650, with an accumulation at the upper limit of 850.
- **Age** is right-skewed, with a median of 37 and a longer tail toward older customers.
- **Tenure** is roughly uniform across years 1–9, with a noticeable dip at year 0 — likely reflecting customers who joined less than a year before the data was collected.
- **Balance** is bimodal: a large spike at 0 (customers with no balance), followed by a roughly normal distribution among customers who do carry a balance — producing a two-part shape.
- **NumOfProducts** is concentrated almost entirely on 1 and 2 products, with very few customers holding 3 or 4.
- **EstimatedSalary** is close to uniform across its full range (0 to 200k), with roughly even counts per bin.

In [ ]:
# Compare the numeric columns by churn status
for column in numeric_columns:
    sns.boxplot(data=df, x='Exited', y=column)
    plt.title(f'{column} by Churn Status')
    plt.show()

**Numerical features vs. churn (boxplots):**

- **Age:** Customers who churned have a noticeably higher median age 45, while the median age is 36 for retained customers — age appears to be a meaningful signal for churn.
- **Tenure:** Distributions are largely similar between the two groups, though Q1 sits slightly lower for churned customers (2 years vs. 3 years for retained customers), suggesting a subset of customers leave relatively early in their tenure.
- **Balance:** A notable difference — customers who stayed have a Q1 of 0 (a large share carry no balance), while customers who churned have a Q1 of approximately $38 000, suggesting that churned customers were more likely to carry a balance.
- `CreditScore` and `EstimatedSalary` showed little separation between churned and retained customers. `NumOfProducts` is examined as a discrete feature in the following section because its relationship with churn is non linear.

## 9. Explore Categorical Features Against Churn

This section examines churn rate broken down by each categorical feature, to identify which customer segments are most likely to leave. Understanding these patterns helps validate whether the categorical features carry meaningful predictive signal ahead of modeling, and starts to build a picture of what an at-risk customer looks like.

In [ ]:
categorical_columns = [
    'Geography', 'Gender', 'NumOfProducts',
    'HasCrCard', 'IsActiveMember'
]

for column in categorical_columns:
    churn_summary = (
        df.groupby(column)['Exited']
        .agg(customer_count='count', churn_rate='mean')
        .sort_values('churn_rate', ascending=False)
    )

    print(f'\n{column}')
    display(churn_summary)

**Findings:**

- **Geography:** Churn rate is notably higher in Germany (32.4%) compared to Spain (16.7%) and France (16.1%), which are similar to each other. Geography — specifically Germany — appears to be a meaningful churn signal.

- **Gender:** Female customers churn at a higher rate (25.0%) than male customers (16.0%).

- **NumOfProducts:** The relationship between product count and churn is strongly nonlinear. Customers with two products have the lowest churn rate, while customers with three or four products have exceptionally high churn rates. However, the three- and four-product groups contain only 266 and 60 customers, so those estimates should be interpreted cautiously.

- **HasCrCard:** No meaningful difference — churn rate is nearly identical for customers without a card (20.8%) and with one (20.2%). This feature does not appear to be a strong churn signal.

- **IsActiveMember:** Inactive customers churn at nearly double the rate of active customers (26.9% vs. 14.3%), making activity status one of the clearer churn signals in the dataset.

## 10. Examine Numerical Relationships

This section examines how numerical features relate to one another and to the target variable (`Exited`), using a correlation matrix. This serves two purposes: identifying features that may carry a linear relationship with churn, and flagging potential multicollinearity between features before modeling.

In [ ]:
correlation_columns = numeric_columns + [
    'HasCrCard', 'IsActiveMember', 'Exited'
]

plt.figure(figsize=(10, 7))
sns.heatmap(
    df[correlation_columns].corr(),
    annot=True,
    cmap='coolwarm',
    center=0
)
plt.title('Feature Correlation Matrix')
plt.show()

**Note:** Correlation captures linear relationships only. A feature may still be predictive of churn through a non-linear pattern even if its correlation coefficient is close to 0 — this is worth keeping in mind when interpreting weak correlations here.

Age has the strongest positive linear association with churn (approximately 0.29), followed by Balance (approximately 0.12). IsActiveMember has the strongest negative association (approximately −0.16). No pair of predictors exhibits severe correlation, so the matrix does not reveal an immediate multicollinearity concern. These values describe linear relationships only.

## 11. Actionable Conclusions

The target variable is imbalanced (79.63% stayed vs. 20.37% churned), so accuracy alone would be misleading for model evaluation — F1 and AUC-ROC remain the primary metrics going forward.

`Tenure` contains approximately 9% missing values. No obvious association was found between its missingness and either churn or geography. Because dropping these records would discard a meaningful portion of the dataset, Tenure will be imputed using the median calculated from the training set. Identifier columns (`RowNumber`, `CustomerId`, and `Surname`) will be excluded during preprocessing.

Several features show a clear association with churn and will be retained for modeling:
- **Age** — churned customers skew notably older (median 45 for churned customers versus 36 for retained customers)
- **Geography** — churn is markedly higher in Germany (32.4%) than Spain or France (~16-17%)
- **IsActiveMember** — inactive customers churn at nearly double the rate of active ones (26.9% vs. 14.3%)
- **NumOfProducts** — churn rises sharply at 3-4 products, a segment worth further investigation given its smaller sample size
- **Balance** — churned customers typically carry a higher balance, while a large share of retained customers carry none

`Geography` and `Gender` will require one-hot encoding before modeling. `HasCrCard` showed no meaningful relationship to churn but will be evaluated during feature selection rather than dropped outright at this stage.

These findings set up the preprocessing steps that follow: encoding categorical features, imputing missing `Tenure` values, and finalizing the feature set for model training.